# Audit: new TS (pn validity mask) vs previous TS vs TI, 1990-2020

Repeats every check from the TS/TI investigation for the new TS version, next to the previous version
(`5_Typology/TS_python_only_v2`, before the pn mask).

Run the cells top to bottom. **Pass A** and **Pass B** each read the rasters once, block by block. On the full
data expect roughly 30-90 minutes in total; both passes print progress.

Outputs (also printed): CSV tables and two maps in `data/5_Typology/audit_pnmask/`.

| Section | Question it answers |
|---|---|
| 0 | Do all files exist, line up, and were the steps run in order? |
| A1 | Footprint and agreement, previous vs new |
| A2 | Class totals (km2) and per-class agreement |
| A3 | Which pixels changed, what they changed to, and why |
| A4 | Why agreement did not go up |
| A5 | Which codes are dropped in the one-sided pixels (incl. code 212) |
| B1 | Do area, edge, and pn now use the same valid pixels? |
| B2 | Python vs TerrSet, per metric |
| B3 | Threshold sweep, per metric |
| B4 | What the TS-only / TI-only pixels look like (z-scores, 1990-2020 change) |
| B5 | How much re-standardising edge moved the +/-0.5 cutoffs |

In [ ]:
### Parameters ###
import os
os.environ["GDAL_CACHEMAX"] = "2048"   # MB; set before rasterio is imported. Lower if RAM is tight.

import csv, math, time
from pathlib import Path
from collections import Counter
import numpy as np, rasterio
from rasterio.windows import Window

D = Path("./data")
OLD_DIR = D / "5_Typology" / "TS_python_only_v2"     # previous TS version (before the pn mask)
F = {
    "old_ts_c":  OLD_DIR / "combined_timeseries" / "combined.tif",
    "old_ts_rc": OLD_DIR / "combined_rc_timeseries" / "combined_rc.tif",
    "new_ts_c":  D / "5_Typology" / "combined_timeseries" / "combined.tif",
    "new_ts_rc": D / "5_Typology" / "combined_rc_timeseries" / "combined_rc.tif",
    "ti_c":      D / "4_TS_and_TI" / "TI" / "1990-2020" / "combined.tif",
    "ti_rc":     D / "5_Typology" / "TimeInterval-1990-2020.tif",
    "mask":      D / "4_TS_and_TI" / "TS_mask" / "pn_validity_mask.tif",
}
METRICS = ("area", "edge", "pn")
TAU  = {m: D / "4_TS_and_TI" / "TS" / m / f"{m}_raster_group_MK_tau.tif" for m in METRICS}
Z    = {m: D / "4_TS_and_TI" / "TS_zscore" / m / f"{m}_MK_tau_z.tif" for m in METRICS}
TERR = {m: D / "4_TS_and_TI" / "terrset_version" / f"{m}_MK_tau_z.tif" for m in METRICS}
DIFF = {m: D / "4_TS_and_TI" / "TI" / "img_diff" / f"90-20_{m}.tif" for m in METRICS}
MW   = {m: sorted((D / "3_MovingWindow" / f"mw_{m}").glob(f"*_{m}_1km.tif")) for m in METRICS}
OUT  = D / "5_Typology" / "audit_pnmask"

THR = 0.5          # standardized-Tau threshold used by the remap
PX_KM2 = 0.0009    # km2 per 30 m pixel
ROWS = 128         # rows per processing strip; lower (e.g. 64) if RAM is tight

# Reference values from the saved outputs of the previous version
REF = {"pn_py_valid": 1_024_652_313, "pn_terr_only": 9_260_941, "pn_r": 1.00000, "pn_rmse": 0.00338}

In [ ]:
### Typology tables and helpers ###
RECODE = {111:1,112:1,113:1, 121:2,122:2,123:2,131:2,132:2,133:2, 211:3, 213:4, 221:5,223:5,
          222:6,220:6, 231:7,232:7,233:7,230:7, 311:8,312:8,313:8, 321:9,322:9,323:9, 331:10,332:10,333:10}
NAMES = ["NoData", "Attrition", "Aggregation", "Shrinkage", "Perforation", "Deformation",
         "Persistent", "Enlargement", "Dissection", "Frag per se", "Creation"]
LUT = np.zeros(1001, np.int16)
for k, v in RECODE.items():
    LUT[k] = v
NC = 1001          # combined-code bins: bin = code + 1 (bin 0 = NoData)


def km2(n):
    return f"{n * PX_KM2:,.0f} km2"


def strips(src):
    for r in range(0, src.height, ROWS):
        yield Window(0, r, src.width, min(ROWS, src.height - r))


def read(src, w):
    """(data, valid) for one window."""
    a = src.read(1, window=w, masked=True)
    return np.ma.getdata(a), ~np.ma.getmaskarray(a)


def read_code(src, w):
    """Combined code as int32; NoData -> -1."""
    d, v = read(src, w)
    return np.where(v, d.astype(np.int32), -1)


def read_class(src, w):
    """Final typology class 1-10 as int16; NoData -> 0."""
    d, v = read(src, w)
    return np.where(v, d.astype(np.int16), 0)


def zero_or_missing(src, w):
    d, v = read(src, w)
    return (~v) | (d == 0)


def combined_class(code):
    return LUT[np.clip(code, 0, 1000)]


def has_zero_digit(c):
    return c > 0 and (c % 10 == 0 or (c // 10) % 10 == 0 or c < 100)


def lbl(c):
    if c < 0:
        return "NoData"
    if c == 0:
        return "0(background)"
    return f"{c}({NAMES[LUT[c]] if LUT[c] else 'dropped'})"


def write_csv(name, header, rows):
    OUT.mkdir(parents=True, exist_ok=True)
    with open(OUT / name, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
    print(f"  saved {OUT / name}")

In [ ]:
### 0) Files, grids, and run order ###
paths = {**F,
         **{f"tau {m}": p for m, p in TAU.items()}, **{f"z {m}": p for m, p in Z.items()},
         **{f"terrset {m}": p for m, p in TERR.items()}, **{f"img_diff {m}": p for m, p in DIFF.items()},
         **{f"mw {m} first": (MW[m][0] if MW[m] else Path("missing")) for m in METRICS},
         **{f"mw {m} last": (MW[m][-1] if MW[m] else Path("missing")) for m in METRICS}}
grid = {}
for name, p in paths.items():
    if Path(p).exists():
        with rasterio.open(p) as s:
            grid[name] = (s.width, s.height)
        t = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"  OK      {name:16} {t}  {p}")
    else:
        print(f"  MISSING {name:16} {p}")
required = ["old_ts_c", "old_ts_rc", "new_ts_c", "new_ts_rc", "ti_c", "ti_rc", "mask"]
missing = [k for k in required if k not in grid]
assert not missing, f"required files missing: {missing}"
for m in METRICS:
    assert MW[m], f"no moving-window rasters found for {m}"
    print(f"  mw {m}: {len(MW[m])} years ({MW[m][0].name} ... {MW[m][-1].name})")
assert len(set(grid.values())) == 1, f"grids differ: {grid}"
print(f"  all {len(grid)} rasters are {next(iter(grid.values()))[0]} x {next(iter(grid.values()))[1]}")
HAVE = {k: k in grid for k in paths}
mt = lambda k: os.path.getmtime(paths[k])
for a, b in (("mask", "z edge"), ("z edge", "new_ts_c"), ("new_ts_c", "new_ts_rc")):
    if HAVE.get(a) and HAVE.get(b):
        ok = mt(b) >= mt(a)
        print(f"  {'OK   ' if ok else 'CHECK'} {b} is {'newer' if ok else 'OLDER'} than {a}")

## Pass A: previous TS vs new TS vs TI
Reads the six typology rasters, the pn mask, the edge 1990/2020 rasters and the TI edge img_diff once.
Writes two maps to `audit_pnmask/`:

**`footprint_cause_new.tif`** (new TS vs TI): 1 = both classified, same class · 2 = both classified, different class ·
3 = TS-only, TI code 212 · 4 = TS-only, other · 5 = TI-only, TS code 212 · 6 = TI-only, other · 0 = neither

**`ts_change_old_vs_new.tif`** (previous TS vs new TS): 1 = newly classified, edge 0 at both endpoints (edge now kept) ·
2 = newly classified, other pixels · 3 = class changed, edge 0 at both endpoints · 4 = class changed, other pixels ·
5 = classified before, NoData now (expected: none) · 0 = no change

In [ ]:
### Pass A ###
conf = {"old": np.zeros(121, np.int64), "new": np.zeros(121, np.int64)}   # TS class x TI class
trans = np.zeros(2 * 11 * 11 * 11, np.int64)        # cause x previous TS x new TS x TI
code_hist = {k: np.zeros(NC, np.int64) for k in ("old_ts", "new_ts", "ti")}
code_trans = np.zeros(NC * NC, np.int64)             # previous -> new TS combined code (changed only)
one_sided = {k: np.zeros(NC, np.int64) for k in
             ("previous TS-only: TI code", "new TS-only: TI code",
              "previous TI-only: TS code", "new TI-only: TS code")}
both212 = {"old": 0, "new": 0}
chk = Counter()          # all expected to be 0
ti_edge = Counter()      # how TI treats edge where edge = 0 at both endpoints
ti_edge_cls = np.zeros(11, np.int64)   # TI class at those pixels

src = {k: rasterio.open(F[k]) for k in ("old_ts_c", "old_ts_rc", "new_ts_c", "new_ts_rc", "ti_c", "ti_rc", "mask")}
src["e0"], src["e1"] = rasterio.open(MW["edge"][0]), rasterio.open(MW["edge"][-1])
src["diff_edge"] = rasterio.open(DIFF["edge"]) if HAVE["img_diff edge"] else None
OUT.mkdir(parents=True, exist_ok=True)
prof = src["new_ts_rc"].profile.copy()
prof.update(driver="GTiff", count=1, dtype="uint8", nodata=0, compress="deflate", tiled=False, blockysize=ROWS)
prof.pop("blockxsize", None)
fp_path, ch_path = OUT / "footprint_cause_new.tif", OUT / "ts_change_old_vs_new.tif"
t0 = time.time()
try:
    with rasterio.open(fp_path, "w", **prof) as fp_dst, rasterio.open(ch_path, "w", **prof) as ch_dst:
        windows = list(strips(src["new_ts_rc"]))
        for i, w in enumerate(windows, 1):
            oc, nc, tc = (read_code(src[k], w) for k in ("old_ts_c", "new_ts_c", "ti_c"))
            ok, nk, tk = (read_class(src[k], w) for k in ("old_ts_rc", "new_ts_rc", "ti_rc"))
            keep = read(src["mask"], w)[0] == 1
            E = zero_or_missing(src["e0"], w) & zero_or_missing(src["e1"], w)   # edge 0 at both endpoints
            ov, nv, tv = ok > 0, nk > 0, tk > 0

            chk["previous final != recode(previous combined)"] += int((combined_class(oc) != ok).sum())
            chk["new final != recode(new combined)"] += int((combined_class(nc) != nk).sum())
            chk["TI final != recode(TI combined)"] += int((combined_class(tc) != tk).sum())
            chk["new TS classified outside the pn mask"] += int((nv & ~keep).sum())
            chk["final classes outside 1-10"] += int(((ok > 10) | (nk > 10) | (tk > 10)).sum())

            for v, k in (("old", ok), ("new", nk)):
                conf[v] += np.bincount((k.astype(np.int64) * 11 + tk).ravel(), minlength=121)
            cause = np.where(E, 0, 1).astype(np.int64)
            trans += np.bincount((((cause * 11 + ok) * 11 + nk) * 11 + tk).ravel(), minlength=trans.size)
            for k, c in (("old_ts", oc), ("new_ts", nc), ("ti", tc)):
                code_hist[k] += np.bincount((np.clip(c, -1, NC - 2) + 1).ravel(), minlength=NC)
            d = oc != nc
            if d.any():
                code_trans += np.bincount((np.clip(oc[d], -1, NC - 2) + 1) * NC + np.clip(nc[d], -1, NC - 2) + 1,
                                          minlength=NC * NC)
            for name, sel, c in (("previous TS-only: TI code", ov & ~tv, tc), ("new TS-only: TI code", nv & ~tv, tc),
                                 ("previous TI-only: TS code", ~ov & tv, oc), ("new TI-only: TS code", ~nv & tv, nc)):
                if sel.any():
                    one_sided[name] += np.bincount(np.clip(c[sel], -1, NC - 2) + 1, minlength=NC)
            both212["old"] += int(((oc == 212) & (tc == 212)).sum())
            both212["new"] += int(((nc == 212) & (tc == 212)).sum())

            sel = E & keep
            ti_edge["edge 0 at both endpoints, inside the pn mask"] += int(sel.sum())
            ti_edge["... TI classified there"] += int((sel & tv).sum())
            ti_edge_cls += np.bincount(tk[sel], minlength=11)
            ti_edge["... TI combined edge digit = 0 (edge treated as NoData)"] += int((sel & (tc > 0) & (tc % 10 == 0)).sum())
            if src["diff_edge"] is not None:
                ti_edge["... TI edge img_diff = NoData"] += int((sel & ~read(src["diff_edge"], w)[1]).sum())

            fpm = np.zeros(nk.shape, np.uint8)
            fpm[nv & tv & (nk == tk)] = 1
            fpm[nv & tv & (nk != tk)] = 2
            fpm[nv & ~tv & (tc == 212)] = 3
            fpm[nv & ~tv & (tc != 212)] = 4
            fpm[~nv & tv & (nc == 212)] = 5
            fpm[~nv & tv & (nc != 212)] = 6
            fp_dst.write(fpm, 1, window=w)
            chm = np.zeros(nk.shape, np.uint8)
            chm[~ov & nv & E] = 1
            chm[~ov & nv & ~E] = 2
            chm[ov & nv & (ok != nk) & E] = 3
            chm[ov & nv & (ok != nk) & ~E] = 4
            chm[ov & ~nv] = 5
            ch_dst.write(chm, 1, window=w)
            if i % max(1, len(windows) // 20) == 0 or i == len(windows):
                print(f"  Pass A: {i:,}/{len(windows):,} strips ({(time.time() - t0) / 60:.1f} min)")
finally:
    for s in src.values():
        if s is not None:
            s.close()
trans = trans.reshape(2, 11, 11, 11)
code_trans = code_trans.reshape(NC, NC)
co, cn = conf["old"].reshape(11, 11), conf["new"].reshape(11, 11)
print(f"maps written:\n  {fp_path}\n  {ch_path}")

### A1) Footprint and agreement
**Look for:** TI valid identical (TI was not rerun) · every consistency check 0 · the agreement change shown to
4 decimals, so the actual difference is visible rather than rounded to 56.8%.

In [ ]:
### A1) Footprint and agreement: previous vs new ###
def footprint(m):
    both, ts_only, ti_only = int(m[1:, 1:].sum()), int(m[1:, 0].sum()), int(m[0, 1:].sum())
    return dict(ts=both + ts_only, ti=both + ti_only, both=both, ts_only=ts_only, ti_only=ti_only,
                nonoverlap=ts_only + ti_only, agree=int(np.trace(m[1:, 1:])))

fo, fn = footprint(co), footprint(cn)
print(f"{'':18}{'previous':>16}{'new':>16}{'change':>14}")
for label, k in (("TS valid", "ts"), ("TI valid", "ti"), ("both valid", "both"), ("TS-only", "ts_only"),
                 ("TI-only", "ti_only"), ("non-overlap", "nonoverlap"), ("agreeing pixels", "agree")):
    print(f"{label:18}{fo[k]:>16,}{fn[k]:>16,}{fn[k] - fo[k]:>+14,}   ({km2(fn[k] - fo[k])})")
ao, an = fo["agree"] / fo["both"], fn["agree"] / fn["both"]
print(f"{'agreement':18}{ao:>16.4%}{an:>16.4%}{(an - ao) * 100:>+13.4f} percentage points")
print("\nTI valid identical:", "OK" if fo["ti"] == fn["ti"] else "CHANGED - check the TI file")
print("\nConsistency checks (all should be 0):")
for k, v in chk.items():
    print(f"  {'OK   ' if v == 0 else 'CHECK'} {k:48} {v:,}")

### A2) Class totals and per-class agreement
*TS-side agreement*: of the pixels TS puts in a class (where TI is classified), the share TI puts in the same class.
*TI-side agreement*: the same from TI's side.

In [ ]:
### A2) Class totals (km2) and per-class agreement ###
print(f"{'class':13}{'TS prev':>10}{'TS new':>10}{'change':>9}{'TI':>10}   "
      f"{'TS-side agreement':>22}   {'TI-side agreement':>22}")
rows = []
for k in range(1, 11):
    ts_o, ts_n, ti = int(co[k, :].sum()), int(cn[k, :].sum()), int(cn[:, k].sum())
    us = [m[k, k] / max(m[k, 1:].sum(), 1) for m in (co, cn)]
    pr = [m[k, k] / max(m[1:, k].sum(), 1) for m in (co, cn)]
    print(f"{NAMES[k]:13}{ts_o * PX_KM2:>10,.0f}{ts_n * PX_KM2:>10,.0f}{(ts_n - ts_o) * PX_KM2:>+9,.0f}"
          f"{ti * PX_KM2:>10,.0f}   {us[0]:>9.1%} -> {us[1]:>7.1%}      {pr[0]:>9.1%} -> {pr[1]:>7.1%}")
    rows.append([NAMES[k], ts_o, ts_n, ti, int(co[k, k]), int(cn[k, k])])
print("(km2)")
write_csv("class_totals_pixels.csv",
          ["class", "TS_previous_px", "TS_new_px", "TI_px", "agree_previous_px", "agree_new_px"], rows)
for v, m in (("previous", co), ("new", cn)):
    write_csv(f"confusion_TS_{v}_vs_TI_pixels.csv", ["TS_class \\ TI_class"] + NAMES,
              [[NAMES[i]] + [int(x) for x in m[i]] for i in range(11)])

### A3) Every pixel whose TS result changed
Split by cause: pixels where **edge is 0 at both 1990 and 2020** (edge was dropped before and is now kept by the pn
mask) vs **all other pixels** (only the edge z-score standardisation changed there). Each line shows how many of
those pixels matched TI before and after.

In [ ]:
### A3) What changed, and why ###
CAUSE = ["edge 0 at both endpoints (edge now kept)", "other pixels (edge z-score shift)"]
for c in range(2):
    on = trans[c].sum(axis=2)            # previous x new
    newly, dropped = int(on[0, 1:].sum()), int(on[1:, 0].sum())
    changed = int(on[1:, 1:].sum() - np.trace(on[1:, 1:]))
    print(f"\n{CAUSE[c]}: newly classified {newly:,} | NoData now {dropped:,} | class changed {changed:,}")
    pairs = sorted(((int(on[o, n]), o, n) for o in range(11) for n in range(11) if o != n and on[o, n]), reverse=True)
    for cnt, o, n in pairs[:12]:
        t = trans[c, o, n]
        print(f"   {NAMES[o]:12} -> {NAMES[n]:12} {cnt:>11,} ({km2(cnt):>9})   matched TI: "
              f"{int(t[o]) if o else 0:,} before -> {int(t[n]) if n else 0:,} now   "
              f"| TI says: " + ", ".join(f"{NAMES[x]} {int(t[x]):,}" for x in np.argsort(t)[::-1][:3] if t[x]))
write_csv("ts_class_changes_by_cause.csv", ["cause", "TS_previous", "TS_new", "TI", "pixels"],
          [[CAUSE[c], NAMES[o], NAMES[n], NAMES[x], int(trans[c, o, n, x])]
           for c in range(2) for o in range(11) for n in range(11) for x in range(11)
           if o != n and trans[c, o, n, x]])

print("\nCombined-code changes, previous TS -> new TS (top 25):")
top = sorted(((int(code_trans[a, b]), a - 1, b - 1) for a, b in np.argwhere(code_trans > 0)), reverse=True)
for n, a, b in top[:25]:
    print(f"   {lbl(a):18} -> {lbl(b):18} {n:>11,}")
write_csv("ts_combined_code_changes.csv", ["previous_code", "new_code", "pixels"], [[a, b, n] for n, a, b in top])

### A4) Why agreement did not go up
Agreement = pixels where TS and TI give the same class ÷ pixels both classify. This splits the net change in
agreeing pixels by cause, and shows how TI treats edge where edge is 0 at both endpoints.

In [ ]:
### A4) Agreement change, by cause ###
o_, n_, t_ = np.meshgrid(np.arange(11), np.arange(11), np.arange(11), indexing="ij")
agree_o, agree_n = (o_ > 0) & (o_ == t_), (n_ > 0) & (n_ == t_)
groups = {"newly classified": (o_ == 0) & (n_ > 0),
          "class changed": (o_ > 0) & (n_ > 0) & (o_ != n_),
          "NoData now": (o_ > 0) & (n_ == 0)}
print(f"{'':58}{'pixels':>12}{'agreed before':>15}{'agree now':>12}{'net':>11}")
net = 0
for c in range(2):
    for g, sel in groups.items():
        t = trans[c]
        px, a0, a1 = int(t[sel].sum()), int(t[sel & agree_o].sum()), int(t[sel & agree_n].sum())
        net += a1 - a0
        if px:
            print(f"{CAUSE[c] + ': ' + g:58}{px:>12,}{a0:>15,}{a1:>12,}{a1 - a0:>+11,}")
print(f"{'net change in agreeing pixels':58}{'':>39}{net:>+11,}")
assert net == fn["agree"] - fo["agree"], "breakdown does not add up"
print(f"\nagreement before: {fo['agree']:,} / {fo['both']:,} = {ao:.4%}")
print(f"agreement now   : {fn['agree']:,} / {fn['both']:,} = {an:.4%}")

print("\nHow TI treats edge where edge = 0 at both 1990 and 2020 (inside the pn mask):")
base = max(ti_edge["edge 0 at both endpoints, inside the pn mask"], 1)
for k, v in ti_edge.items():
    print(f"  {k:58} {v:>14,}  ({v / base:.1%})")
t = ti_edge_cls
print("  TI class at these pixels: " + ", ".join(f"{NAMES[x]} {int(t[x]):,}" for x in np.argsort(t)[::-1] if t[x]))

### A5) One-sided pixels and dropped codes
Equivalent of the earlier cells 7, 8, 9, 15 and 20, for the previous and the new TS.

In [ ]:
### A5) Which codes are dropped? ###
for name, h in one_sided.items():
    tot = int(h.sum())
    print(f"\n{name}: {tot:,} px ({km2(tot)})")
    for b in np.argsort(h)[::-1][:8]:
        if h[b] == 0:
            break
        print(f"   {lbl(b - 1):18} {int(h[b]):>12,} ({km2(int(h[b]))})")

print("\nCode 212:")
for v, key in (("previous", "old_ts"), ("new", "new_ts")):
    vv = "old" if v == "previous" else "new"
    print(f"  {v:8} TS 212 {int(code_hist[key][213]):>12,} ({km2(int(code_hist[key][213]))}) | "
          f"TI 212 {int(code_hist['ti'][213]):,} | 212 in both {both212[vv]:,} ({km2(both212[vv])})")

print("\nCombined codes dropped by the recode map, whole map (background 0 excluded):")
for key, label in (("old_ts", "previous TS"), ("new_ts", "new TS"), ("ti", "TI")):
    h = code_hist[key]
    dropped = sorted(((int(h[c + 1]), c) for c in range(1, 1000) if h[c + 1] and LUT[c] == 0), reverse=True)
    zero = sum(n for n, c in dropped if has_zero_digit(c))
    print(f"  {label:12} {sum(n for n, _ in dropped):>12,} px | with a 0 digit {zero:>9,} | "
          + ", ".join(f"{c}: {n:,}" for n, c in dropped[:6]))

for label, row in (("TI class of new TI-only pixels", cn[0, :]), ("TS class of new TS-only pixels", cn[:, 0])):
    print(f"\n{label}: " + ", ".join(f"{NAMES[k]} {int(row[k]):,}" for k in np.argsort(row)[::-1] if k and row[k]))

## Pass B: per-metric checks
For each of area, edge, and pn, reads the Tau, z-score, TerrSet, TI img_diff, and 1990/2020 moving-window rasters once.
Groups (TS-only, TI-only, both) use the **new** TS final typology vs TI.

In [ ]:
### Pass B ###
SWEEP = (0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5)
GROUPS = ("both valid", "TS-only", "TI-only")
B = {}
for m in METRICS:
    if not (HAVE[f"tau {m}"] and HAVE[f"z {m}"]):
        print(f"{m}: Tau or z-score missing, skipped")
        continue
    r = B[m] = dict(tau_total=0, tau_valid=0, tau_zero=0, tau_outside_mask=0, z_valid=0, z_vs_tau=0,
                    z_sum=0.0, z_sq=0.0, terr=Counter() if HAVE[f"terrset {m}"] else None,
                    sweep=np.zeros((len(SWEEP), 2), np.int64), groups={g: Counter() for g in GROUPS},
                    edge=dict(n_all=0, s_all=0.0, q_all=0.0, n_E=0, s_E=0.0, q_E=0.0, band=Counter()))
    s = {"tau": rasterio.open(TAU[m]), "z": rasterio.open(Z[m]), "mask": rasterio.open(F["mask"]),
         "ts": rasterio.open(F["new_ts_rc"]), "ti": rasterio.open(F["ti_rc"]),
         "m0": rasterio.open(MW[m][0]), "m1": rasterio.open(MW[m][-1])}
    if r["terr"] is not None:
        s["terr"] = rasterio.open(TERR[m])
    if HAVE[f"img_diff {m}"]:
        s["diff"] = rasterio.open(DIFF[m])
    t0 = time.time()
    try:
        windows = list(strips(s["tau"]))
        for i, w in enumerate(windows, 1):
            tau, tv = read(s["tau"], w); tau = tau.astype(np.float64)
            z, zv = read(s["z"], w); z = z.astype(np.float64)
            keep = read(s["mask"], w)[0] == 1
            ts_ok, ti_ok = read(s["ts"], w)[1], read(s["ti"], w)[1]
            a0, v0 = read(s["m0"], w); a1, v1 = read(s["m1"], w)
            a0, a1 = a0.astype(np.float64), a1.astype(np.float64)

            r["tau_total"] += tau.size
            r["tau_valid"] += int(tv.sum())
            r["tau_zero"] += int((tv & (tau == 0)).sum())
            r["tau_outside_mask"] += int((tv & ~keep).sum())
            r["z_valid"] += int(zv.sum())
            r["z_vs_tau"] += int((zv != tv).sum())
            r["z_sum"] += float(z[zv].sum()); r["z_sq"] += float((z[zv] ** 2).sum())

            if "terr" in s:
                y, yv = read(s["terr"], w); y = y.astype(np.float64)
                T, b = r["terr"], zv & yv
                T["py_only"] += int((zv & ~yv).sum()); T["terr_only"] += int((~zv & yv).sum()); T["both"] += int(b.sum())
                T["ty_n"] += int(yv.sum()); T["ty_s"] += float(y[yv].sum()); T["ty_q"] += float((y[yv] ** 2).sum())
                if b.any():
                    x_, y_ = z[b], y[b]
                    dd = x_ - y_
                    T["sd"] += float(dd.sum()); T["sd2"] += float((dd * dd).sum())
                    T["max"] = max(T["max"], float(np.abs(dd).max()))
                    T["sx"] += float(x_.sum()); T["sy"] += float(y_.sum())
                    T["sxx"] += float((x_ * x_).sum()); T["syy"] += float((y_ * y_).sum()); T["sxy"] += float((x_ * y_).sum())

            if "diff" in s:
                dv, dvv = read(s["diff"], w)
                sel = zv & dvv
                if sel.any():
                    ti_dir, zs = np.sign(dv[sel]), z[sel]
                    for k, thr in enumerate(SWEEP):
                        ts_dir = np.where(zs > thr, 1, np.where(zs < -thr, -1, 0))
                        r["sweep"][k, 0] += zs.size
                        r["sweep"][k, 1] += int((ts_dir == ti_dir).sum())

            ends = v0 & v1
            for g, gm in zip(GROUPS, (ts_ok & ti_ok, ts_ok & ~ti_ok, ~ts_ok & ti_ok)):
                G = r["groups"][g]
                G["px"] += int(gm.sum())
                G["z_nodata"] += int((gm & ~zv).sum())
                gz = gm & zv
                if gz.any():
                    zz = z[gz]
                    G["z_n"] += zz.size; G["z_sum"] += float(zz.sum())
                    G["z_stable"] += int((np.abs(zz) <= THR).sum())
                    G["z_inc"] += int((zz > THR).sum()); G["z_dec"] += int((zz < -THR).sum())
                ge = gm & ends
                if ge.any():
                    net, st, en = (a1 - a0)[ge], a0[ge], a1[ge]
                    G["e_n"] += net.size; G["net_sum"] += float(net.sum()); G["abs_sum"] += float(np.abs(net).sum())
                    G["nochange"] += int((net == 0).sum())
                    G["small"] += int(((st > 0) & (np.abs(net) < 0.1 * st)).sum())
                    G["zero_both"] += int(((st == 0) & (en == 0)).sum())

            if m == "edge":
                E = (~v0 | (a0 == 0)) & (~v1 | (a1 == 0))
                Ed = r["edge"]
                x, xe = tau[tv], tau[tv & E]
                Ed["n_all"] += x.size; Ed["s_all"] += float(x.sum()); Ed["q_all"] += float((x * x).sum())
                Ed["n_E"] += xe.size; Ed["s_E"] += float(xe.sum()); Ed["q_E"] += float((xe * xe).sum())
                band = tv & ~E & (((tau > 0.2) & (tau < 0.4)) | ((tau > -0.3) & (tau < -0.1)))
                if band.any():
                    vals, cnt = np.unique(tau[band].astype(np.float32), return_counts=True)
                    for v_, c_ in zip(vals, cnt):
                        Ed["band"][float(v_)] += int(c_)
            if i % max(1, len(windows) // 10) == 0 or i == len(windows):
                print(f"  Pass B {m}: {i:,}/{len(windows):,} strips ({(time.time() - t0) / 60:.1f} min)")
    finally:
        for x_ in s.values():
            x_.close()

### B1) Tau and z-score footprint per metric
**Look for:** the same Tau-valid count for all three metrics, 0 outside the mask, pn unchanged from the previous run.

In [ ]:
### B1) Tau / z-score footprint ###
print(f"{'metric':7}{'Tau valid':>16}{'Tau exactly 0':>15}{'NoData':>8}{'outside mask':>14}"
      f"{'z valid':>16}{'z vs Tau':>10}{'z mean':>9}{'z std':>8}")
for m, r in B.items():
    n = max(r["z_valid"], 1); zm = r["z_sum"] / n; zs = math.sqrt(max(r["z_sq"] / n - zm * zm, 0))
    print(f"{m:7}{r['tau_valid']:>16,}{r['tau_zero']:>15,}{1 - r['tau_valid'] / r['tau_total']:>8.1%}"
          f"{r['tau_outside_mask']:>14,}{r['z_valid']:>16,}{r['z_vs_tau']:>10,}{zm:>+9.4f}{zs:>8.4f}")
same = len({r["tau_valid"] for r in B.values()}) == 1
print("\n" + ("OK: all metrics use the same valid pixels" if same else "CHECK: valid pixel counts differ between metrics"))
if "pn" in B:
    print(f"pn Tau valid {B['pn']['tau_valid']:,} vs previous {REF['pn_py_valid']:,}: "
          f"{'unchanged' if B['pn']['tau_valid'] == REF['pn_py_valid'] else 'CHANGED'}")

In [ ]:
### B2) Python standardized Tau vs TerrSet ###
for m, r in B.items():
    T = r["terr"]
    if T is None or not T["both"]:
        print(f"{m}: no TerrSet file or no overlap, skipped")
        continue
    n = T["both"]
    den = math.sqrt((n * T["sxx"] - T["sx"] ** 2) * (n * T["syy"] - T["sy"] ** 2))
    rr = (n * T["sxy"] - T["sx"] * T["sy"]) / den if den else float("nan")
    tm = T["ty_s"] / T["ty_n"]; tsd = math.sqrt(max(T["ty_q"] / T["ty_n"] - tm * tm, 0))
    print(f"{m}: Python {r['z_valid']:,} | TerrSet {T['ty_n']:,} | both {n:,} | "
          f"Python-only {T['py_only']:,} | TerrSet-only {T['terr_only']:,}")
    print(f"      TerrSet mean {tm:+.4f} std {tsd:.4f} | mean diff {T['sd'] / n:+.5f} | "
          f"RMSE {math.sqrt(T['sd2'] / n):.5f} | max |diff| {T['max']:.5f} | r {rr:.5f}")
print(f"\npn previous: Python {REF['pn_py_valid']:,} | TerrSet-only {REF['pn_terr_only']:,} | "
      f"r {REF['pn_r']:.5f} | RMSE {REF['pn_rmse']}")
print("edge now keeps full-forest-window pixels (edge = 0 at both endpoints), so its footprint vs TerrSet is expected to change.")

In [ ]:
### B3) Threshold sweep: TS direction vs TI img_diff sign ###
print(f"{'threshold':>10}" + "".join(f"{m:>10}" for m in B))
for k, thr in enumerate(SWEEP):
    print(f"{'+/-' + str(thr):>10}" + "".join(
        f"{(r['sweep'][k, 1] / r['sweep'][k, 0] if r['sweep'][k, 0] else float('nan')):>10.1%}" for r in B.values()))

In [ ]:
### B4) What the TS-only / TI-only / both-valid pixels look like ###
for g in GROUPS:
    print(f"\n=== {g} (new TS vs TI) ===")
    for m, r in B.items():
        G = r["groups"][g]
        if not G["px"]:
            continue
        zn, en = max(G["z_n"], 1), max(G["e_n"], 1)
        print(f"  {m:4} {G['px']:,} px ({km2(G['px'])}) | z mean {G['z_sum'] / zn:+.3f}: stable {G['z_stable'] / zn:.1%}, "
              f"increase {G['z_inc'] / zn:.1%}, decrease {G['z_dec'] / zn:.1%}, z NoData {G['z_nodata']:,}")
        print(f"       1990->2020: mean net {G['net_sum'] / en:+.2f}, mean |net| {G['abs_sum'] / en:.2f}, "
              f"no change {G['nochange'] / en:.1%}, |net| < 10% of 1990 {G['small'] / en:.1%}, 0 at both ends {G['zero_both']:,}")

In [ ]:
### B5) Edge: how far re-standardising moved the +/-0.5 cutoffs ###
if "edge" in B:
    Ed = B["edge"]["edge"]
    n_new, n_old = Ed["n_all"], Ed["n_all"] - Ed["n_E"]
    m_new = Ed["s_all"] / n_new; sd_new = math.sqrt(Ed["q_all"] / n_new - m_new ** 2)
    m_old = (Ed["s_all"] - Ed["s_E"]) / n_old; sd_old = math.sqrt((Ed["q_all"] - Ed["q_E"]) / n_old - m_old ** 2)
    up_o, up_n, lo_o, lo_n = m_old + THR * sd_old, m_new + THR * sd_new, m_old - THR * sd_old, m_new - THR * sd_new
    print(f"edge Tau previous: n={n_old:,} mean={m_old:.6f} std={sd_old:.6f} (rebuilt)")
    print(f"edge Tau new     : n={n_new:,} mean={m_new:.6f} std={sd_new:.6f}")
    print(f"edge kept by the pn mask (0 at both endpoints): {Ed['n_E']:,} ({km2(Ed['n_E'])})")
    print(f"'increase' cutoff in raw Tau: {up_o:.5f} -> {up_n:.5f}")
    print(f"'decrease' cutoff in raw Tau: {lo_o:.5f} -> {lo_n:.5f}")
    fu = {v: c for v, c in Ed["band"].items() if (v > up_o) != (v > up_n)}
    fl = {v: c for v, c in Ed["band"].items() if (v < lo_o) != (v < lo_n)}
    print(f"{'stable -> increase' if up_n < up_o else 'increase -> stable'}: {sum(fu.values()):,} px "
          f"at Tau {', '.join(f'{v:.5f} (= {v * 465:.1f}/465)' for v in sorted(fu))}")
    print(f"{'stable -> decrease' if lo_n > lo_o else 'decrease -> stable'}: {sum(fl.values()):,} px "
          + (f"at Tau {', '.join(f'{v:.5f}' for v in sorted(fl))}" if fl else ""))

In [ ]:
### Summary: previous vs new ###
h_o, h_n = code_hist["old_ts"], code_hist["new_ts"]
zero_o = sum(int(h_o[c + 1]) for c in range(1, 1000) if LUT[c] == 0 and has_zero_digit(c))
zero_n = sum(int(h_n[c + 1]) for c in range(1, 1000) if LUT[c] == 0 and has_zero_digit(c))
rows = [("TS valid px", fo["ts"], fn["ts"]), ("TI valid px", fo["ti"], fn["ti"]),
        ("both valid px", fo["both"], fn["both"]), ("TS-only px", fo["ts_only"], fn["ts_only"]),
        ("TI-only px", fo["ti_only"], fn["ti_only"]), ("non-overlap px", fo["nonoverlap"], fn["nonoverlap"]),
        ("agreeing px", fo["agree"], fn["agree"]),
        ("TS code 212 px", int(h_o[213]), int(h_n[213])),
        ("TS codes with a 0 digit px", zero_o, zero_n),
        ("TS Persistent px", int(co[6, :].sum()), int(cn[6, :].sum())),
        ("TS Deformation px", int(co[5, :].sum()), int(cn[5, :].sum())),
        ("TS Perforation px", int(co[4, :].sum()), int(cn[4, :].sum()))]
print(f"{'':28}{'previous':>16}{'new':>16}{'change':>14}")
for label, a, b in rows:
    print(f"{label:28}{a:>16,}{b:>16,}{b - a:>+14,}")
print(f"{'agreement':28}{ao:>16.4%}{an:>16.4%}{(an - ao) * 100:>+13.4f} pts")
write_csv("summary_previous_vs_new.csv", ["measure", "previous", "new"], [[l, a, b] for l, a, b in rows])